In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules
from mlxtend.frequent_patterns import fpgrowth
import matplotlib.pyplot as plt

In [ ]:
import random  # Set global random seed

def set_global_seed(seed_value): 
    random.seed(seed_value)        # Set Python random seed
    np.random.seed(seed_value)     # Set NumPy random seed

set_global_seed(66)

In [ ]:
data = pd.read_csv("xxx.CSV")  # 

In [ ]:
# node
node = [
        'CLL0', 'CLL1',
        'CI1', 'CI0', 'CI2',
        'CDL7' ,'CDL2', 'CDL3' ,'CDL1', 'CDL4', 'CDL6', 'CDL5',
        'IT1', 'IT3', 'IT4', 'IT15', 'IT13', 'IT11', 'IT14', 'IT5', 'IT9', 'IT7', 'IT2', 'IT8', 'IT10', 'IT6', 'IT12',
        'POBI6', 'POBI1', 'POBI3', 'POBI2', 'POBI4', 'POBI7', 'POBI9', 'POBI5', 'POBI8',
        'ROCP8', 'ROCP7', 'ROCP3', 'ROCP9', 'ROCP6', 'ROCP4', 'ROCP2', 'ROCP11', 'ROCP5', 'ROCP1', 'ROCP10',
        'AOCP6', 'AOCP1', 'AOCP2', 'AOCP3', 'AOCP4', 'AOCP5',
        'CD6', 'CD8', 'CD5', 'CD7', 'CD4', 'CD9', 'CD3', 'CD10', 'CD2' ,'CD1',
        'ST4', 'ST3', 'ST1', 'ST2', 'ST5',
        'GT6', 'GT1', 'GT5', 'GT2', 'GT4', 'GT3', 'GT7',
        'SO2' ,'SO1', 'SO5', 'SO15', 'SO9', 'SO3', 'SO10', 'SO7' ,'SO6', 'SO8', 'SO4' ,'SO14', 'SO13', 'SO11', 'SO12',
        'POB3', 'POB11', 'POB8', 'POB4', 'POB13' ,'POB10' ,'POB5', 'POB2', 'POB6', 'POB9', 'POB12', 'POB7' ,'POB1',
        'NL1', 'NL2' ,'NL4' ,'NL3',
        'SS4', 'SS2', 'SS1', 'SS9', 'SS3', 'SS5', 'SS7', 'SS6' ,'SS8',
        'V2', 'V1', 'V3', 'V6' ,'V5', 'V4',
        'WF4', 'WF3', 'WF2', 'WF7', 'WF12', 'WF1', 'WF6', 'WF5', 'WF8', 'WF9', 'WF11', 'WF10',
        'W1', 'W4', 'W3', 'W2'
        ]

# ARM

In [ ]:
def arm(database, min_sup, min_con):
    """
    Function: Association rule mining
    Input: minimum support, minimum confidence
    Output: association rules
    """
    tran = []
    for i in range(database.shape[0]):
        tran.append([str(database.values[i, j]) for j in range(database.shape[1])])

    tran = np.array(tran)
    te = TransactionEncoder()
    tran_te = te.fit(tran).transform(tran)
    tran_df = pd.DataFrame(tran_te, columns=te.columns_)

    frequent_items = fpgrowth(tran_df, min_support=min_sup, use_colnames=True, max_len=2)

    rules = association_rules(frequent_items, metric='lift', min_threshold=1, num_itemsets=10)
    result = rules.sort_values("confidence", ascending=False)  # Sort by confidence
    result = result[result["confidence"] >= min_con]
    result = result[result["lift"] > 1]

    return result

In [ ]:
ar = arm(data, min_sup=0.1, min_con=0.5)  # Minimum confidence and minimum support can be adjusted

In [ ]:
#data change
name=["antecedents","consequents","confidence"]
ar=ar[name]
ar=ar.rename(columns={"antecedents":"source","consequents":"target","confidence":"weight"})

In [ ]:
def list_to_matrix(data):
    """
    Function: Convert association rule data into a matrix
    """
    G = nx.DiGraph(data)
    A = nx.to_numpy_array(G)
    print('1111111')
    return G, A, np.array(list(G.nodes()))  # Return complex network, adjacency matrix, and corresponding nodes

In [ ]:
G,A,n = list_to_matrix(ar)

In [ ]:
def node_tran(data, node):
    """
    Function: Node transformation (nodes in association rules may be frozen and need to be unfrozen)
    Input: nodes output when association rules are converted into a matrix, manually input nodes
    Output: unfrozen nodes
    """
    new_node = []
    for no in data:
        for nd in node:
            if no == frozenset({nd}):
                new_node.append(nd)
    return np.array(new_node)

In [ ]:
n_new = node_tran(n,node)

# WINGS

In [ ]:
def influence_matrix(data, cn, node):
    """
    Compute the direct influence matrix (add diagonal values to the complex network matrix)
    Diagonal values are set as support values.
    Input: original data, original complex network matrix, nodes corresponding to the complex network
    """
    im = cn.copy()

    for i in range(len(im)):
        nod = node[i]
        sup = 0
        for j in range(len(data)):
            for k in range(len(data.iloc[0])):
                if data.iloc[j][k] == nod:
                    sup += 1
        im[i, i] = (sup / len(data))

    return im

In [ ]:
im = influence_matrix(data, cn=A, node=n_new)

In [ ]:
def Normalized_influence_matrix(data):
    """
    Normalize the direct influence matrix; input data is the direct influence matrix
    """
    # Compute row sums
    a = []
    for i in data:
        a.append(np.sum(i))

    # Normalization parameter
    para = np.sum(a)

    nor_ma = data / para

    return nor_ma

In [ ]:
nm = Normalized_influence_matrix(im)

In [ ]:
def com_influence_matrix(data):
    """
    Convert the normalized matrix into a comprehensive influence matrix
    """
    E = np.eye(len(data))

    T1 = np.linalg.inv(E - data)
    T = np.dot(data, T1)

    return T

In [ ]:
cim = com_influence_matrix(nm)

In [ ]:
def WINGS_index(data):
    """
    Calculate influence degree, affected degree, centrality, cause degree, and weights of factors
    """
    # Influence degree
    D = []
    for i in data:
        D.append(np.sum(i))

    # Affected degree
    C = []
    for i in range(len(data)):
        cc = 0
        for j in range(len(data)):
            cc += data[j][i]
        C.append(cc)

    # Centrality
    M = []
    for i in range(len(data)):
        M.append(D[i] + C[i])

    # Cause degree
    R = []
    for i in range(len(data)):
        R.append(D[i] - C[i])

    # Weights
    w = np.zeros(len(data))
    ww = 0
    for i in range(len(data)):
        ww1 = (M[i] ** 2 + R[i] ** 2) ** (1 / 2)
        ww += ww1
        w[i] = ww1
    w = w / ww

    return D, C, M, R, list(w)

In [ ]:
D, C, M, R, W = WINGS_index(cim)

In [ ]:
D=pd.DataFrame(D)
C=pd.DataFrame(C)
M=pd.DataFrame(M)
R=pd.DataFrame(R)
W=pd.DataFrame(W)
name=pd.DataFrame(node)
F=[name,D,C,M,R,W]
data_export=pd.concat(F,axis=1)
data_export.to_csv('WINGS.csv')

# AISM

In [ ]:
def lamda(data):
    """
    Intercept
    Input: comprehensive influence matrix
    """
    x = np.mean(data)

    summ = 0
    num = len(data)
    for i in range(num):
        for j in range(num):
            summ += (data[i, j] - x) ** 2
    sigma = (summ / (num ** 2)) ** (1 / 2)

    return x + sigma

In [ ]:
def Relationship_Matrix(data, lamda):
    """
    Relationship matrix A

    Input: comprehensive influence matrix, intercept
    """
    A = np.zeros((len(data), len(data)))

    for i in range(len(data)):
        for j in range(len(data)):
            if data[i, j] > lamda:
                A[i, j] = 1

    return A

In [ ]:
RM = Relationship_Matrix(data=cim, lamda=lamda(cim))

In [ ]:
def Boolean_operation(A, B):
    """
    Boolean matrix multiplication
    """
    bol = np.zeros((len(A), len(B[0])))

    for i in range(len(A)):
        for j in range(len(B[0])):
            num = 0
            for m in range(len(B)):
                if A[i, m] >= 1 and B[m, j] >= 1:
                    num += 1
            if num > 0:
                bol[i, j] = 1
    # Diagonal elements are always 1
    return bol

In [ ]:
def reachable_matrix(data):
    """
    Compute the reachability matrix
    Input: relationship matrix
    """
    E = np.eye(len(data))
    B = data + E

    B1 = B
    B2 = np.zeros((len(data), len(data)))
    h = np.ones(len(data))
    while np.linalg.norm((B1 - B2)) != 0:
        B2 = B1
        B1 = Boolean_operation(B1, B)

    R = B1
    return R

In [ ]:
Reach = reachable_matrix(RM)

In [ ]:
def tarjan(graph):
    """
    Function: Find strongly connected components
    Input: reachability matrix
    Output: indices of strongly connected components
    """
    n = len(graph)
    index_counter = [0]
    index = [-1] * n
    lowlink = [-1] * n
    onStack = [False] * n
    stack = []

    result = []

    def strongconnect(v):
        index[v] = index_counter[0]
        lowlink[v] = index_counter[0]
        index_counter[0] += 1
        stack.append(v)
        onStack[v] = True

        for w in range(n):
            if graph[v][w] == 1:
                if index[w] == -1:
                    strongconnect(w)
                    lowlink[v] = min(lowlink[v], lowlink[w])
                elif onStack[w]:
                    lowlink[v] = min(lowlink[v], index[w])

        if lowlink[v] == index[v]:
            scc = []
            while True:
                w = stack.pop()
                onStack[w] = False
                scc.append(w)
                if w == v:
                    break
            result.append(scc)

    for v in range(n):
        if index[v] == -1:
            strongconnect(v)

    return result


In [ ]:
strcon = tarjan(Reach)

In [ ]:
strcon

In [ ]:
st_con=[strcon[0],strcon[13],strcon[20]]

In [ ]:
def shrink(R, st_con, lst):
    """
    Compute the condensed (shrinked) matrix by contracting strongly connected components into a single node
    Input: reachability matrix, strongly connected components, list of node names
    """
    n1 = len(st_con)
    n2 = 0
    for i in st_con:
        n2 += len(i)
    n3 = len(R)

    n = n3 - n2 + n1

    S = np.zeros((n, n))  # Condensed matrix initialized with all zeros

    ind = []
    for i in range(n3):
        num = 0
        for j in st_con:
            if i in j:
                num += 1
        if num == 0:
            ind.append(i)

    # Construct new matrix
    for i in range(n):

        if i < len(ind):  # Non-strongly connected region
            for j in range(n):
                if j < len(ind):
                    S[i, j] = R[ind[i], ind[j]]
                else:
                    ss = 0
                    for m in st_con[j - len(ind)]:
                        ss += R[ind[i], m]
                    if ss == 0:
                        S[i, j] = 0
                    else:
                        S[i, j] = 1

        else:  # Strongly connected region
            for j in range(n):
                if j < len(ind):
                    ss = 0
                    for m in st_con[i - len(ind)]:
                        ss += R[m, ind[j]]

                    if ss == 0:
                        S[i, j] = 0
                    else:
                        S[i, j] = 1
                else:

                    ss = 0
                    for m in st_con[i - len(ind)]:
                        for k in st_con[j - len(ind)]:
                            ss += R[m, k]
                    if ss == 0:
                        S[i, j] = 0
                    else:
                        S[i, j] = 1

    # Reset diagonal of condensed nodes to zero

    # Return labels
    labels = []
    for i in ind:
        labels.append(lst[i])
    for i in range(n1):
        labels.append("Shrink" + str(i + 1))
    return S, labels


In [ ]:
SN, labels = shrink(Reach, st_con, n_new)

In [ ]:
def s_edge(SN):
    """
    Shrink-edge matrix, derive the skeleton matrix
    """
    E = np.eye(len(SN))
    B = Boolean_operation((SN - E), (SN - E))
    SE = SN - B - E

    return SE

In [ ]:
SE = s_edge(SN)

In [ ]:
def show(data, lst):
    """
    Intuitively display which nodes are connected
    Input: skeleton matrix, labels
    """
    n = len(data)
    sh = []
    for i in range(n):
        ss = ["Start " + lst[i]]
        for j in range(n):
            if data[i, j] == 1:
                ss.append(lst[j])
        if len(ss) == 1:
            ss.append(None)
        sh.append(ss)
    return sh


In [ ]:
sh=show(SE, labels)

In [ ]:
sh

In [ ]:
def RS_A(data, lis):
    """
    Reachability set
    Input: skeleton matrix, column labels of the matrix
    """
    RS = []
    for i in range(len(data)):
        ss = []
        for j in range(len(data)):
            if data[i, j] == 1:
                ss.append(lis[j])
        RS.append(ss)
    return RS

In [ ]:
def QS_A(data, lis):
    """
    Antecedent set
    Input: skeleton matrix, column labels of the matrix
    """
    QS = []
    for i in range(len(data)):
        ss = []
        for j in range(len(data)):
            if data[j, i] == 1:
                ss.append(lis[j])
        QS.append(ss)
    return QS

In [ ]:
def TS_A(RS, QS):
    """
    Intersection of reachability set and antecedent set
    """
    TS = []
    for i in range(len(RS)):
        ss = []
        for j in RS[i]:
            if j in QS[i]:
                ss.append(j)
        TS.append(ss)
    return TS

In [ ]:
def up_and_down_extract(data, lis):
    """
    Up-type and Down-type extraction
    Input: reachability matrix, column labels of the matrix
    """
    up = []
    down = []
    R = data.copy()

    RS = RS_A(R, lis)
    QS = QS_A(R, lis)
    TS1 = TS_A(RS, QS)
    TS2 = TS_A(RS, QS)
    num = np.sum(R)

    while num != 0:

        # Up-type
        uup = []
        for i in range(len(RS)):
            if len(RS[i]) != 0:
                if RS[i] == TS1[i]:
                    for j in RS[i]:
                        uup.append(j)  # Store labels

        uup = list(set(uup))
        up.append(uup)

        # Update up table
        RS_n = []
        for i in RS:
            ss = []
            for j in i:
                if j not in uup:
                    ss.append(j)
            RS_n.append(ss)
        RS = RS_n

        TS1_n = []
        for i in TS1:
            ss = []
            for j in i:
                if j not in uup:
                    ss.append(j)
            TS1_n.append(ss)
        TS1 = TS1_n

        # Down-type
        ddon = []
        for i in range(len(QS)):
            if len(QS[i]) != 0:
                if QS[i] == TS2[i]:
                    for j in QS[i]:
                        ddon.append(j)  # Store labels

        ddon = list(set(ddon))
        down.append(ddon)

        # Update down table
        QS_n = []
        for i in QS:
            ss = []
            for j in i:
                if j not in ddon:
                    ss.append(j)
            QS_n.append(ss)
        QS = QS_n

        TS2_n = []
        for i in TS2:
            ss = []
            for j in i:
                if j not in ddon:
                    ss.append(j)
            TS2_n.append(ss)
        TS2 = TS2_n

        # Check whether all elements have been extracted
        num = 0
        for i in RS:
            num += len(i)
        for i in QS:
            num += len(i)

    # Up-type
    up
    # Down-type (reverse the order)
    down_n = []
    for i in range(len(down)):
        down_n.append(down[-(i + 1)])
    down = down_n

    return up, down

In [ ]:
up, down = up_and_down_extract(Reach,n_new)

In [ ]:
# Up-type hierarchical structure
up

In [ ]:
# down-type hierarchical structure
down